YAML

---
title: Calculadora de Columnas

description: Análisis de esfuerzo-deformación (Modelo de Mander)

by: Arq. Julia Patricia Villegas Ayala

show-code: false

params:

    t_analisis:
        label: "Modelo:"
        value: Columna
        choices: ["Concreto simple", "Columna"]
    t_columna:
        label: "Geometría:"
        value: Rectangular
        choices: ["Circular", "Rectangular"]
    tipo_calc:
        label: "Tipo de Cálculo:"
        value: Ambos
        choices: ["Estático", "Dinámico", "Ambos"]
    fco_val:
        label: "f'co (kg/cm²):"
        value: 350
        input: numeric
    diam_val:
        label: "D (cm):"
        value: 40
        input: numeric
    b_val:
        label: "b (cm):"
        value: 40
        input: numeric
    h_val:
        label: "h (cm):"
        value: 30
        input: numeric
    recub:
        label: "Recubrimiento (cm):"
        value: 5
        input: numeric
    t_conf_val:
        label: "Confinamiento:"
        value: Estribo
        choices: ["Estribo", "Espiral"]
    v_trans:
        label: "Varilla Transversal:"
        value: "#3"
        choices: ["#2", "#2.5", "#3", "#4", "#5", "#6", "#8"]
    s_val:
        label: "Separación s (cm):"
        value: 20
        input: numeric
    v_long:
        label: "Varilla Longitudinal:"
        value: "#6"
        choices: ["#3","#4","#5","#6","#8","#9","#10","#12"]
    n_barras_val:
        label: "Cantidad total barras:"
        value: 8
        input: numeric
    n_x_mer:
        label: "Barras en cara X:"
        value: 4
        input: numeric
    n_y_mer:
        label: "Barras en cara Y:"
        value: 2
        input: numeric
    r_x:
        label: "Ramas en X (v):"
        value: 3.66
        input: numeric
    r_y:
        label: "Ramas en Y (h):"
        value: 4.0
        input: numeric
    comp_nc:
        label: "Comparar con no confinado"
        value: true
        input: checkbox
---

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

In [14]:
# OUTPUT
out = widgets.Output()

# BOTONES
tipo_analisis = widgets.ToggleButtons(
    options=["Concreto simple", "Columna"],
    description="Modelo:"
)

tipo_columna = widgets.ToggleButtons(
    options=["Circular", "Rectangular"],
    description="Tipo:",
    layout=widgets.Layout(display='none')
)

tipo_calculo = widgets.ToggleButtons(
    options=["Estático", "Dinámico", "Ambos"],
    description="Cálculo:",
    layout=widgets.Layout(display='none')
)

comparar = widgets.Checkbox(
    value=False,
    description="Comparar con no confinado",
    layout=widgets.Layout(display='none')
)

# INPUTS
fco = widgets.FloatText(description="f'co (kg/cm²):")

diametro = widgets.FloatText(description="D (cm):", layout=widgets.Layout(display='none'))
b = widgets.FloatText(description="b (cm):", layout=widgets.Layout(display='none'))
h = widgets.FloatText(description="h (cm):", layout=widgets.Layout(display='none'))

recubrimiento = widgets.FloatText(
    description="Recubrimiento c (cm):",
    layout=widgets.Layout(display='none')
)

tipo_confinamiento = widgets.ToggleButtons(
    options=["Estribo", "Espiral"],
    description="Confinamiento:",
    layout=widgets.Layout(display='none')
)

varilla_transv = widgets.Dropdown(
    options=["#2","#2.5","#3","#4","#5","#6","#8"],
    description="Varilla Transversal:",
    layout=widgets.Layout(display='none')
)

separacion = widgets.FloatText(description="s (cm):", layout=widgets.Layout(display='none'))

varilla_long = widgets.Dropdown(
    options=["#3","#4","#5","#6","#8","#9","#10","#12"],
    description="Varilla Longitudinal:",
    layout=widgets.Layout(display='none')
)

num_barras = widgets.IntText(description="Cantidad barras:", layout=widgets.Layout(display='none'))

nx = widgets.FloatText(description="Barras en X:", layout=widgets.Layout(display='none'))
ny = widgets.FloatText(description="Barras en Y:", layout=widgets.Layout(display='none'))

ramas_x = widgets.FloatText(description="Ramas en X:", layout=widgets.Layout(display='none'))
ramas_y = widgets.FloatText(description="Ramas en Y:", layout=widgets.Layout(display='none'))

boton_calcular = widgets.Button(description="CALCULAR", button_style='success')

# VISIBILIDAD
def mostrar_opciones(change):

    if tipo_analisis.value == "Columna":
        tipo_columna.layout.display = 'block'
        # Forzar actualización cambio de sin confinar a columnas
        actualizar_geometria(None)

        tipo_calculo.layout.display = 'block'
        tipo_confinamiento.layout.display = 'block'
        varilla_transv.layout.display = 'block'
        separacion.layout.display = 'block'
        varilla_long.layout.display = 'block'
        num_barras.layout.display = 'block'
        comparar.layout.display = 'block'
        recubrimiento.layout.display = 'block'
        ramas_x.layout.display = 'block'
        ramas_y.layout.display = 'block'
    else:
        tipo_columna.layout.display = 'none'
        tipo_calculo.layout.display = 'none'
        tipo_confinamiento.layout.display = 'none'
        varilla_transv.layout.display = 'none'
        separacion.layout.display = 'none'
        varilla_long.layout.display = 'none'
        num_barras.layout.display = 'none'
        diametro.layout.display = 'none'
        b.layout.display = 'none'
        h.layout.display = 'none'
        comparar.layout.display = 'none'
        nx.layout.display = 'none'
        ny.layout.display = 'none'
        recubrimiento.layout.display = 'none'
        ramas_x.layout.display = 'none'
        ramas_y.layout.display = 'none'

tipo_analisis.observe(mostrar_opciones, names='value')


def actualizar_geometria(change):

    if tipo_columna.value == "Circular":
        diametro.layout.display = 'block'
        b.layout.display = 'none'
        h.layout.display = 'none'
        tipo_confinamiento.options = ["Espiral", "Estribo"]

        nx.layout.display = 'none'
        ny.layout.display = 'none'

        ramas_x.layout.display = 'none'
        ramas_y.layout.display = 'none'

    else:
        diametro.layout.display = 'none'
        b.layout.display = 'block'
        h.layout.display = 'block'
        tipo_confinamiento.options = ["Estribo"]

        nx.layout.display = 'block'
        ny.layout.display = 'block'

        ramas_x.layout.display = 'block'
        ramas_y.layout.display = 'block'

tipo_columna.observe(actualizar_geometria, names='value')

In [16]:
# --- CELDA DE CONEXIÓN MERCURY ---
tipo_analisis.value = globals().get('t_analisis', "Columna")
tipo_columna.value = globals().get('t_columna', "Rectangular")
tipo_calculo.value = globals().get('tipo_calc', "Ambos")
fco.value = globals().get('fco_val', 350)
diametro.value = globals().get('diam_val', 40)
b.value = globals().get('b_val', 40)
h.value = globals().get('h_val', 30)
recubrimiento.value = globals().get('recub', 5)
tipo_confinamiento.value = globals().get('t_conf_val', "Estribo")
varilla_transv.value = globals().get('v_trans', "#3")
separacion.value = globals().get('s_val', 20)
varilla_long.value = globals().get('v_long', "#6")
num_barras.value = int(globals().get('n_barras_val', 8))

# Conexión para el dibujo y acero longitudinal
nx.value = int(globals().get('n_x_mer', 4))
ny.value = int(globals().get('n_y_mer', 2))

# Conexión para los decimales del Mander
ramas_x.value = float(globals().get('r_x', 3.66))
ramas_y.value = float(globals().get('r_y', 4.0))

comparar.value = globals().get('comp_nc', True)

# IMPORTANTE: Disparar el cálculo manual para Mercury
# Esto asegura que la gráfica salga al cargar la página
if 'calcular' in globals():
    calcular(None)

In [15]:
# CÁLCULO
def calcular(boton):

    with out:
        clear_output()

        # Validación de f'co
        if fco.value <= 0:
            print("❌ Error: f'co inválido")
            return

        fco_mpa = fco.value / 10.197
        epsilon = np.linspace(0, 0.02, 500)

        # CONCRETO SIMPLE
        if tipo_analisis.value == "Concreto simple":

            eco = 0.002
            e2 = 2 * eco
            esp = 0.005

            Ec = 4700 * np.sqrt(fco_mpa)
            r = Ec / (Ec - fco_mpa/eco)

            fc = np.zeros_like(epsilon)

            x2 = e2 / eco
            f2 = (fco_mpa * x2 * r) / (r - 1 + x2**r)

            for i, e in enumerate(epsilon):
                if e <= e2:
                    x = e / eco
                    fc[i] = (fco_mpa * x * r) / (r - 1 + x**r)
                elif e <= esp:
                    fc[i] = f2 * (1 - (e - e2) / (esp - e2))
                else:
                    fc[i] = np.nan

            #Deformación ultima
            idx_pico_nc = np.nanargmax(fc)
            deformacion_pico_nc = epsilon[idx_pico_nc]
            esfuerzo_pico_nc = fc[idx_pico_nc] * 10.197 # Convertido a kg/cm²

            print(f"εcu: {deformacion_pico_nc:.5f}")
            print(f"Esfuerzo máximo: {esfuerzo_pico_nc:.2f} kg/cm²")

            plt.figure()
            plt.plot(epsilon, fc*10.197, label="No confinado")
            plt.legend()
            plt.grid()
            plt.show()

        # COLUMNA
        if tipo_analisis.value == "Columna":
            if separacion.value <= 0:
              print("❌ Error: separación inválida")
              return
            if num_barras.value <= 0:
              print("❌ Error: número de barras inválido")
              return

            tabla_phi = {
                "#2":0.64, "#2.5":0.79, "#3":0.95, "#4":1.27,
                "#5":1.59, "#6":1.91, "#8":2.54,
                "#9":2.87, "#10":3.22, "#12":3.81
            }

            # Diamétros
            phi = tabla_phi[varilla_transv.value]
            phi_long = tabla_phi[varilla_long.value]

            s = separacion.value
            n = num_barras.value

            # Áreas
            Asp = np.pi * phi**2 / 4
            As_barra = np.pi * phi_long**2 / 4
            Asl = n * As_barra



            fy = 4200 / 10.197
            Ec = 4700 * np.sqrt(fco_mpa)
            eco = 0.002

            # COLUMNA CIRCULAR
            if tipo_columna.value == "Circular":

                D = diametro.value
                c = recubrimiento.value

                if c <= 0:
                     print("❌ Error: recubrimiento inválido")
                     return
                if c < 2:
                    print("⚠️ Advertencia: recubrimiento muy pequeño")

                dc = D - 2*c - phi

                if dc <= 0:
                  print("❌ Error: núcleo inválido")
                  return

                Asl = n * As_barra

                Ag = np.pi * D**2 / 4
                rho_g = Asl / Ag
                porcentaje_acero = rho_g * 100

                print("\n--- VALIDACIÓN DE DISEÑO ---")
                print(f"Porcentaje de acero longitudinal: {porcentaje_acero:.2f}%")
                if porcentaje_acero < 1.0:
                    print("❌ RECHAZADO: El acero es menor al 1% mínimo normativo.")
                elif porcentaje_acero > 8.0:
                    print("❌ RECHAZADO: El acero supera el 8% máximo normativo.")
                elif porcentaje_acero > 6.0:
                    print("⚠️ ADVERTENCIA: Acero alto (riesgo de congestión al colar).")
                else:
                    print("✅ APTO: Cuantía de acero dentro de los límites.")

                rho_s = (4 * Asp) / (dc * s)
                rho_sl = Asl / (np.pi * dc**2 / 4)

                if tipo_confinamiento.value == "Espiral":
                    ke = (1 - s/(2*dc)) / (1 - rho_sl)
                else:
                     ke = (1 - s/(2*dc)) / (1 - rho_sl)


            # COLUMNA RECTANGULAR
            else:
                B = b.value
                H = h.value
                c = recubrimiento.value

                if c <= 0:
                     print("❌ Error: recubrimiento inválido")
                     return
                if c < 2:
                    print("⚠️ Advertencia: recubrimiento muy pequeño")

                n_ramas_v = ramas_x.value
                n_ramas_h = ramas_y.value

                if n_ramas_v <= 0 or n_ramas_h <= 0:
                    print("❌ Error: El número de ramas debe ser mayor a 0")
                    return

                # Áreas
                Asl = n * As_barra
                Ash = (n_ramas_h + n_ramas_v) * Asp

                # Núcleo confinado
                bc = B - 2*c
                hc = H - 2*c

                if bc <= 0 or hc <= 0:
                    print("❌ Error: dimensiones inválidas del núcleo")
                    return

                # Cuantías
                rho_x = (n_ramas_h * Asp) / (s * bc)
                rho_y = (n_ramas_v * Asp) / (s * hc)
                rho_s = (rho_x + rho_y) / 2

                # Validación
                if rho_s <= 0:
                    print("❌ Error: cuantía transversal inválida")
                    return

                # Cuantía longitudinal
                rho_cc = Asl / (bc * hc)

                Ag = B * H
                rho_g = Asl / Ag
                porcentaje_acero = rho_g * 100

                print("\n--- VALIDACIÓN DE DISEÑO ---")
                print(f"Porcentaje de acero longitudinal: {porcentaje_acero:.2f}%")
                if porcentaje_acero < 1.0:
                    print("❌ RECHAZADO: El acero es menor al 1% mínimo normativo.")
                elif porcentaje_acero > 8.0:
                    print("❌ RECHAZADO: El acero supera el 8% máximo normativo.")
                elif porcentaje_acero > 6.0:
                    print("⚠️ ADVERTENCIA: Acero alto (riesgo de congestión al colar).")
                else:
                    print("✅ APTO: Cuantía de acero dentro de los límites.")

                # Eficiencia de confinamiento
                ke = (1 - s/(2*bc)) * (1 - s/(2*hc))

            # CONFINAMIENTO
            fl = ke * rho_s * fy

            fcc = fco_mpa * (2.254*np.sqrt(1+7.94*fl/fco_mpa) - 2*(fl/fco_mpa) -1.254)
            ecc = eco * (1 + 5*(fcc/fco_mpa -1))

            den = Ec - fcc/ecc
            if abs(den) < 1e-6:
                den = 1e-6

            r = Ec / den

            fc_conf = (fcc*(epsilon/ecc)*r)/(r-1+(epsilon/ecc)**r)

            # Corte de curva
            fc_conf[epsilon > 0.015] = np.nan

            # NO CONFINADO
            e2 = 2 * eco
            esp = 0.005

            Ec_nc = 4700 * np.sqrt(fco_mpa)
            r_nc = Ec_nc / (Ec_nc - fco_mpa/eco)

            fc_nc = np.zeros_like(epsilon)

            x2 = e2 / eco
            f2 = (fco_mpa * x2 * r_nc) / (r_nc - 1 + x2**r_nc)

            for i, e in enumerate(epsilon):
                if e <= e2:
                    x = e / eco
                    fc_nc[i] = (fco_mpa * x * r_nc) / (r_nc - 1 + x**r_nc)
                elif e <= esp:
                    fc_nc[i] = f2 * (1 - (e - e2) / (esp - e2))
                else:
                    fc_nc[i] = np.nan

            # Con Taza de deformación tipica para sismos
            fc_dyn = None  # Para evitar error

            if tipo_calculo.value in ["Dinámico","Ambos"]:

                strain_rate = 0.01

                Df = ((1+(strain_rate/(0.035*(fco_mpa**2)))**(1/6)) /
                      (1+(0.00001/(0.035*(fco_mpa**2)))**(1/6)))

                fcc_dyn = fcc * Df

                fc_dyn = (fcc_dyn*(epsilon/ecc)*r)/(r-1+(epsilon/ecc)**r)

                # Corte de curva
                fc_dyn[epsilon > 0.015] = np.nan

                print("\n--- RESULTADOS CON TAZA DE DEFORMACIÓN EN SISMOS ---")
                print(f"f'cc Dinámico = {fcc_dyn:.3f} MPa")

            # RESULTADOS COLUMNA CIRCULAR
            if tipo_columna.value == "Circular":

              print("\n--- RESULTADOS COLUMNA CIRCULAR ---")
              print(f"Asl = {Asl:.2f} cm²")
              print(f"ρs = {rho_s:.4f}")
              print(f"ρsl = {rho_sl:.4f}")
              print(f"fl = {fl:.3f} MPa")
              print(f"f'cc = {fcc:.3f} MPa")


            # RESULTADOS COLUMNA RECTANGULAR
            else:

              print("\n--- RESULTADOS COLUMNA RECTANGULAR ---")
              print(f"As = {Asl:.2f} cm²")
              print(f"Ash = {Ash:.2f} cm²")
              print(f"ρcc = {rho_cc:.4f}")
              print(f"ρs = {rho_s:.4f}")
              print(f"fl = {fl:.3f} MPa")
              print(f"f'cc = {fcc:.3f} MPa")

            print("\n--- DEFORMACIONES Y DUCTILIDAD ---")

            # Función interna para calcular pico, rama descendente y ductilidad
            def calcular_ductilidad(eps, esfuerzos):
                idx_pico = np.nanargmax(esfuerzos)
                deformacion_pico = eps[idx_pico]
                esfuerzo_max = esfuerzos[idx_pico]

                # El criterio de falla usual es cuando cae al 85% de la resistencia máxima
                esfuerzo_falla = 0.85 * esfuerzo_max

                # Buscamos el punto de falla (εcu) en la rama descendente
                deformacion_ultima = eps[-1] # Por defecto, el último punto

                for i in range(idx_pico, len(esfuerzos)):
                    # Si llegamos al límite de la gráfica (los np.nan que pusiste a los 0.015)
                    if np.isnan(esfuerzos[i]):
                        deformacion_ultima = eps[i-1]
                        break
                    # Si el esfuerzo cae al 85% o menos
                    if esfuerzos[i] <= esfuerzo_falla:
                        deformacion_ultima = eps[i]
                        break

                ductilidad = deformacion_ultima / deformacion_pico if deformacion_pico > 0 else 0
                return deformacion_pico, deformacion_ultima, ductilidad

            # Evaluamos e imprimimos según lo que el usuario haya seleccionado
            if comparar.value:
                ep_pico, ep_ult, duct = calcular_ductilidad(epsilon, fc_nc)
                print(f"No confinado:   εcc = {ep_pico:.5f} | εcu = {ep_ult:.5f} | Ductilidad (μ) = {duct:.2f}")

            if tipo_calculo.value in ["Estático", "Ambos"]:
                ep_pico, ep_ult, duct = calcular_ductilidad(epsilon, fc_conf)
                print(f"Conf. Estático: εcc = {ep_pico:.5f} | εcu = {ep_ult:.5f} | Ductilidad (μ) = {duct:.2f}")

            if fc_dyn is not None:
                ep_pico, ep_ult, duct = calcular_ductilidad(epsilon, fc_dyn)
                print(f"Conf. Dinámico: εcc = {ep_pico:.5f} | εcu = {ep_ult:.5f} | Ductilidad (μ) = {duct:.2f}")

            # GRÁFICA
            plt.figure()

            if comparar.value:
                plt.plot(epsilon, fc_nc*10.197, '--', label="No confinado")

            if tipo_calculo.value in ["Estático","Ambos"]:
                plt.plot(epsilon, fc_conf*10.197, label="Confinado estático")

            if fc_dyn is not None:
               plt.plot(epsilon, fc_dyn*10.197, label="Confinado dinámico")

            plt.xlabel("Deformación")
            plt.ylabel("Esfuerzo (kg/cm²)")

            if tipo_columna.value == "Circular":
                plt.title("Columna circular")
            else:
                plt.title("Columna rectangular")

            plt.legend()
            plt.grid()
            plt.xlim(0, 0.02)

            plt.show()

# EVENTO
boton_calcular.on_click(calcular)

if not os.environ.get('MERCURY_WIDGETS'):
   # DISPLAY
   display(
       tipo_analisis,
       tipo_columna,
       tipo_calculo,
       fco,
       diametro,
       b,
       h,
       recubrimiento,
       tipo_confinamiento,
       varilla_transv,
       separacion,
       varilla_long,
       num_barras,
       nx,
       ny,
       ramas_x,
       ramas_y,
       comparar,
       boton_calcular,
       out
   )

calcular(None)

ToggleButtons(description='Modelo:', options=('Concreto simple', 'Columna'), value='Concreto simple')

ToggleButtons(description='Tipo:', layout=Layout(display='none'), options=('Circular', 'Rectangular'), value='…

ToggleButtons(description='Cálculo:', layout=Layout(display='none'), options=('Estático', 'Dinámico', 'Ambos')…

FloatText(value=0.0, description="f'co (kg/cm²):")

FloatText(value=0.0, description='D (cm):', layout=Layout(display='none'))

...

FloatText(value=0.0, description='Ramas en X:', layout=Layout(display='none'))

FloatText(value=0.0, description='Ramas en Y:', layout=Layout(display='none'))

Checkbox(value=False, description='Comparar con no confinado', layout=Layout(display='none'))

Button(button_style='success', description='CALCULAR', style=ButtonStyle())

Output()